# CTDDG · Five-model molecular evaluation · 10 independent runs
### Reproducible comparison · Paper Tables 1–3 · Two distinct validity methods

**Read-only inputs:** generated molecules already saved by the MXNet and PyTorch sampling notebooks. **Outputs:** a new `Evaluation_Results` directory inside `Validity_Analysis`.

**Supported model folders:** `nitesh_model`, `Pretrained_no_toxicity`, `Pretrained_with_toxicity`, `Pretrained_clean_5epoch`, and `Finetuned_BindingDB_ctddg`. This notebook **does not load generator weights, regenerate molecules, use protein embeddings, or alter sample files**. For the three unconditional PyTorch models, protein IDs are sampling *slots*, not conditioning. The new BindingDB fine-tuned model is protein-conditioned on actual saved protein vectors; this is not the same as the legacy MXNet model.

**Execution order:** configuration → file inventory → sanity checks → RDKit properties → both validity definitions → uniqueness/novelty → Ames predictions → Table 1 & 3 comparison → exports/plots/audit.

> ⚠️ Results are not supplied in this notebook: it must run on Bhavani with your actual generated CSVs and optional property-predictor/reference files. No missing metric is replaced with a fabricated value.

## 1 · Audit: what the references actually calculate

- **CTDDG paper (Table 1, page 11):** validity, uniqueness, logP < 5, exact molecular weight < 500, H-bond donors < 5, acceptors < 10, rotatable bonds < 10, TPSA < 140, mean QED, and SAS < 6; 104 test proteins; **10 runs × 100 generated samples**. This notebook uses the strict `<` thresholds printed in the table.
- **Table 2 (page 12):** percent of ligands with SMINA docking affinity `< -7.0 kcal/mol` for five named proteins (PDB: `4nos`, `2nru`, `1lqf`, `1k2r`, `1cqp`). Not calculable from SMILES alone; optional externally computed docking CSV supported below.
- **Table 3 (page 13):** predicted Ames-mutagenic percent for the same five named proteins; assessed using **pkCSM**, averaged over 10 runs. An independent RF Ames classifier is **not the same method**.
- **Table 4 (page 14):** CDK2 case study, includes SMINA binding energies, QED, SAS, Lipinski violations, pkCSM rat LD50 and fathead minnow LC50; these are not automatically reproducible from the current three model sample folders.
- **Mentor's `Evaluation_metrics.ipynb`:** labels `N` as `beam_size`, but computes *validity* as `100 × exported CSV row count / N` after generation has filtered samples. It does **not** validate each row in that formula. Uniqueness is `100 × distinct raw SMILES / N`. For threshold properties it uses `100 × (distinct raw SMILES/N) × (number passing/N)`; QED is `sum(QED)/N`. It averages across proteins within each run and takes the **population SD of run-level means**. The mentor notebook's 10-protein loading loop conflicts with its later 104-protein indexing; its original data paths also do not apply here.

**Two required validity reports**:

- **Method A · mentor-style exported-yield percentage:** `100 × exported rows / requested N`. Its denominator is the target sample budget, *not* actual generation attempts. It can equal 100% even when many unsuccessful graphs were discarded. Values >100% are flagged; not clamped.
- **Method B · independent RDKit raw-attempt validity:** `100 × attempts producing RDKit-parseable, sanitizable, nonempty molecules / all recorded graph attempts`. Requires `*_raw_attempts.csv`. **MXNet Method B = N/A** if its generator never saved failed attempts. A third column, **RDKit validity among exported SMILES**, is supplied as a diagnostic but *must not* be mislabelled as Method B.

Neither mentor `beam_size` nor our per-protein sampling budget proves beam-search decoding; your code uses stochastic sampling until a quota of accepted molecules is reached.

## 2 · Configuration (the only cell you normally need to edit)

In [ ]:
from pathlib import Path
import os

ROOT = Path(os.environ.get('CTDDG_ROOT', '/home/muhamed/repo/CTDDG'))
REWORKED = ROOT / 'REWORKED_CODE'
VALIDITY_ROOT = Path(os.environ.get('CTDDG_VALIDITY_ROOT', REWORKED / 'testing' / 'Validity_Analysis'))
GENERATED_ROOT = VALIDITY_ROOT / 'Generated_104_10runs'
OUTPUT_DIR = Path(os.environ.get('CTDDG_EVAL_OUTPUT', VALIDITY_ROOT / 'Evaluation_Results_5models_10runs'))

MODELS = ('nitesh_model', 'Pretrained_no_toxicity', 'Pretrained_with_toxicity',
          'Pretrained_clean_5epoch', 'Finetuned_BindingDB_ctddg')
PYTORCH_MODELS = frozenset(MODELS) - {'nitesh_model'}
MODEL_IS_PROTEIN_CONDITIONAL = {
    'nitesh_model': True,
    'Pretrained_no_toxicity': False,
    'Pretrained_with_toxicity': False,
    'Pretrained_clean_5epoch': False,
    'Finetuned_BindingDB_ctddg': True,
}
EXPECTED_PROTEIN_IDS = [f'protein{i}' for i in range(1, int(os.environ.get('CTDDG_EXPECTED_PROTEINS', '104')) + 1)]
SAMPLE_BUDGET = int(os.environ.get('CTDDG_SAMPLE_BUDGET', '100'))  # mentor's N; NOT raw attempts
PAPER_RUNS = 10
STRICT_COMPLETENESS = True  # ensure all 104 protein folders are present before evaluation

# Novelty against a VERIFIED training/reference ligand set only. Never compare against a guessed dataset.
NOVELTY_REFERENCE_FILE = None  # optional common comparison corpus for all five models
NOVELTY_REFERENCE_BY_MODEL = {}  # optional: {'nitesh_model': Path(...), ...} for true model-training novelty
NOVELTY_SMILES_COLUMN = 'SMILES'  # change to match the explicit reference CSV

# Ames source: RF predictor for all five models, as requested; no pkCSM substitution.
AMES_MODE = 'rf_if_available'
RF_MODEL_FILE = REWORKED / 'Random_Forest' / 'RF_NItesh.joblib'  # inspect actual file/case
RF_FINGERPRINT_MODE = 'morgan_bits'    # VERIFY with the RF training script
RF_RADIUS, RF_BITS, RF_POSITIVE_LABEL = 2, 2048, 1
RF_FEATURE_CONTRACT_VERIFIED = False  # True only after confirming training features AND class semantics
PKCSM_AMes_CSV = None   # offline pkCSM export; column names configured below
PKCSM_SMILES_COLUMN, PKCSM_LABEL_COLUMN = 'SMILES', 'Ames toxicity'

# ONLY set this mapping after independently identifying PDB IDs in your actual BindingDB split.
# Example syntax (DO NOT guess mappings): {'4nos':'protein42', ...}
PDB_TO_PROTEIN_ID = {}

# Optional docking results, already computed with SMINA: model, protein_id, Run_ID, SMILES, binding_kcal_mol
DOCKING_CSV = None

WRITE_DETAILED_ROWS = True
PLOT_RESULTS = True
REQUIRE_TEN_COMPLETE_RUNS = True  # enforce 10 x 100 rows per protein and matching raw logs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Inputs :', GENERATED_ROOT)
print('Outputs:', OUTPUT_DIR)
print('Expected IDs:', len(EXPECTED_PROTEIN_IDS), '| budget per run:', SAMPLE_BUDGET)

## 3 · Dependencies and exact input/output boundaries

In [ ]:
import sys, re, json, hashlib, warnings, math, statistics
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDConfig, DataStructs, RDLogger, rdBase
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED, rdMolDescriptors

RDLogger.DisableLog('rdApp.error')
print('Python:', sys.version.split()[0], '| RDKit:', rdBase.rdkitVersion,
      '| pandas:', pd.__version__, '| numpy:', np.__version__)
print('No MXNet or PyTorch is needed: this notebook reads generated CSVs only.')
if not GENERATED_ROOT.is_dir():
    raise FileNotFoundError(f'Missing generation root: {GENERATED_ROOT}. Run on Bhavani or set CTDDG_VALIDITY_ROOT.')

def sha256(path):
    h=hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(2**20), b''): h.update(part)
    return h.hexdigest()

# Use RDKit's maintained contributed SAS scoring implementation (Ertl–Schuffenhauer).
import importlib.util
sas_path = Path(RDConfig.RDContribDir) / 'SA_Score' / 'sascorer.py'
if not sas_path.is_file():
    raise FileNotFoundError('RDKit SA_Score/sascorer.py missing. Install a full RDKit build with Contrib; do not set SAS=0.')
spec = importlib.util.spec_from_file_location('ctddg_sascorer', sas_path)
sascorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sascorer)
print('SAS implementation:', sas_path)

## 4 · Discover actual model files (no filename guessing)

Expected, as shown in your terminal session:

```text
Validity_Analysis/Generated_104_10runs/<model>/proteinN/proteinN_all_samples.csv
```

For the four PyTorch models we additionally look **beside** each main CSV for `proteinN_raw_attempts.csv` and `proteinN_generation_info.json`. A missing raw log is recorded as missing; we **do not invent failures or a sample denominator**. Only CSVs with the exact `_all_samples.csv` suffix enter the main analysis.

In [ ]:
file_rows=[]
for model in MODELS:
    model_folder=GENERATED_ROOT / model
    for protein in EXPECTED_PROTEIN_IDS:
        folder=model_folder/protein
        main=folder/f'{protein}_all_samples.csv'
        raw=folder/f'{protein}_raw_attempts.csv'
        info=folder/f'{protein}_generation_info.json'
        file_rows.append(dict(model=model, protein_id=protein,
            main_csv=str(main), main_exists=main.is_file(),
            raw_csv=str(raw), raw_exists=raw.is_file(),
            metadata_json=str(info), metadata_exists=info.is_file(),
            main_bytes=main.stat().st_size if main.is_file() else 0,
            main_sha256=sha256(main) if main.is_file() else None,
            raw_sha256=sha256(raw) if raw.is_file() else None))
inventory=pd.DataFrame(file_rows)
display(inventory.groupby('model')[['main_exists','raw_exists','metadata_exists']].sum().astype(int))

# Read-only provenance audit. Missing legacy metadata remains a warning, but the
# new fine-tuned model must explicitly declare real protein conditioning.
for row in inventory[inventory.metadata_exists].itertuples(index=False):
    record=json.loads(Path(row.metadata_json).read_text())
    if row.model=='Finetuned_BindingDB_ctddg':
        if record.get('conditioned_on_protein') is not True:
            raise ValueError(f'Fine-tuned samples do not certify protein conditioning: {row.metadata_json}')
        if record.get('fine_tuning_provenance')!='exploratory_leaked_pretraining':
            print('WARNING: fine-tuning provenance differs from supplied training notebook:',
                  record.get('fine_tuning_provenance'))
    elif row.model in PYTORCH_MODELS and record.get('conditioned_on_protein',False):
        raise ValueError(f'Unconditional model has conditional sample metadata: {row.metadata_json}')
missing=inventory[~inventory.main_exists]
if len(missing):
    print(f'WARNING: {len(missing)} of {len(inventory)} expected main CSVs are missing. Showing first 12:')
    display(missing[['model','protein_id','main_csv']].head(12))
    if STRICT_COMPLETENESS:
        raise FileNotFoundError('STRICT_COMPLETENESS=True: generate all requested model samples first, including the fine-tuned model')
if inventory.main_exists.sum()==0:
    raise FileNotFoundError('No generated sample CSVs found. Check Generated_104 folder and model names.')
# Extra folders/files are informational, not silently ingested.
for model in MODELS:
    base=GENERATED_ROOT/model
    extra=[str(p) for p in base.glob('protein*/protein*_all_samples.csv')
           if re.fullmatch(r'protein\d+',p.parent.name)
           and p.parent.name not in EXPECTED_PROTEIN_IDS]
    if extra: print(f'WARNING: {model}: {len(extra)} extra protein CSVs outside expected IDs:',extra[:3])
inventory.to_csv(OUTPUT_DIR/'input_file_inventory.csv', index=False)
print('Sample layouts checked for five models. All sample files are read-only.')

## 5 · Read main CSVs, preserving model / protein / run and all duplicate rows

In [ ]:
def _get_column(frame, name, alternate=()):
    lower={str(c).strip().lower(): c for c in frame.columns}
    for n in (name,*alternate):
        if n.lower() in lower: return lower[n.lower()]
    return None

def read_main_file(row):
    f=Path(row.main_csv)
    df=pd.read_csv(f, keep_default_na=False)
    smile_col=_get_column(df,'SMILES',('smiles','generated_smiles'))
    run_col=_get_column(df,'Run_ID',('run_id','run'))
    if smile_col is None or run_col is None:
        raise ValueError(f'{f}: expected Run_ID,SMILES; found {list(df.columns)}')
    if df.empty:
        raise ValueError(f'{f}: empty CSV; preserve partial run and investigate before evaluation.')
    runs=pd.to_numeric(df[run_col],errors='coerce')
    if runs.isna().any() or (runs<=0).any() or (runs%1!=0).any():
        raise ValueError(f'{f}: Run_ID must contain positive integer runs.')
    return pd.DataFrame({'model':row.model, 'protein_id':row.protein_id,
        'run_id':runs.astype(int), 'smiles':df[smile_col].fillna('').astype(str).str.strip(),
        'source_file':str(f), 'source_row':np.arange(1,len(df)+1,dtype=int)})

parts=[read_main_file(row) for row in inventory[inventory.main_exists].itertuples(index=False)]
samples=pd.concat(parts,ignore_index=True)
samples.insert(0,'sample_row_id',np.arange(len(samples),dtype=np.int64))
main_counts=samples.groupby(['model','run_id','protein_id'],sort=False).size().rename('exported_rows').reset_index()
excess=main_counts[main_counts.exported_rows>SAMPLE_BUDGET]
short=main_counts[main_counts.exported_rows<SAMPLE_BUDGET]
if len(excess): print('WARNING: exported rows exceed mentor budget:',len(excess)); display(excess.head())
if len(short): print('WARNING: incomplete exports below budget:',len(short)); display(short.head())
print('Actual exported molecule rows:',len(samples))
display(samples.groupby(['model','run_id']).agg(protein_slots=('protein_id','nunique'), exported_rows=('smiles','size')))
display(samples.head(6))

### 5a · Ten-run experiment completeness gate

The evaluation must not silently pool a partial run or compare 1-run and 10-run outputs. Before calculating metrics, confirm exactly 104 protein slots × 10 run IDs × 100 exported molecules **for each model**. Set `REQUIRE_TEN_COMPLETE_RUNS=False` only for an explicitly labelled *exploratory partial* run.

In [ ]:
# Validate exactly 104 proteins × 10 run IDs × 100 filtered molecules per model.
expected_groups={(m,p,r) for m in MODELS for p in EXPECTED_PROTEIN_IDS for r in range(1,PAPER_RUNS+1)}
observed_groups=set(map(tuple, main_counts[['model','protein_id','run_id']].itertuples(index=False,name=None)))
absent=sorted(expected_groups-observed_groups)
extra=sorted(observed_groups-expected_groups)
counts_wrong=main_counts.loc[main_counts.exported_rows.ne(SAMPLE_BUDGET)]
print(f'Expected {len(MODELS)} × {len(EXPECTED_PROTEIN_IDS)} × {PAPER_RUNS} =', len(expected_groups), 'protein-run groups')
print('Observed:',len(observed_groups),'| missing:',len(absent),'| unexpected:',len(extra),
      '| incomplete/oversized:',len(counts_wrong))
if absent:print('First missing:',absent[:6])
if extra:print('First unexpected:',extra[:6])
if len(counts_wrong):display(counts_wrong.head(6))
if REQUIRE_TEN_COMPLETE_RUNS and (absent or extra or len(counts_wrong)):
    raise ValueError('The 10-run generation corpus is incomplete/inconsistent; '
                     'inspect inventory and finish generation before reporting.')

## 6 · Read raw attempt logs for four PyTorch models; MXNet logs optional

In [ ]:
def read_raw_file(row):
    p=Path(row.raw_csv)
    frame=pd.read_csv(p,keep_default_na=False)
    expected={'Run_ID','Attempt_ID','SMILES','decode_status','saved_in_main_csv'}
    if not expected.issubset(frame.columns):
        raise ValueError(f'{p}: raw-attempt format not recognized, missing {sorted(expected-set(frame.columns))}')
    r=pd.to_numeric(frame['Run_ID'],errors='coerce')
    a=pd.to_numeric(frame['Attempt_ID'],errors='coerce')
    if r.isna().any() or a.isna().any(): raise ValueError(f'{p}: invalid Run_ID or Attempt_ID')
    accepted=frame['saved_in_main_csv'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    if accepted.isna().any(): raise ValueError(f'{p}: saved_in_main_csv must be bool-like')
    return pd.DataFrame({'model':row.model,'protein_id':row.protein_id,'run_id':r.astype(int),
        'attempt_id':a.astype(int), 'smiles':frame['SMILES'].fillna('').astype(str).str.strip(),
        'decode_status':frame['decode_status'].astype(str), 'saved_in_main_csv':accepted.astype(bool),
        'conversion_error':frame['conversion_error'].astype(str) if 'conversion_error' in frame else '',
        'raw_file':str(p)})

raw_parts=[read_raw_file(row) for row in inventory[inventory.raw_exists].itertuples(index=False)]
raw=pd.concat(raw_parts,ignore_index=True) if raw_parts else pd.DataFrame(columns=[
    'model','protein_id','run_id','attempt_id','smiles','decode_status','saved_in_main_csv','conversion_error','raw_file'])
raw.insert(0,'raw_row_id',np.arange(len(raw),dtype=np.int64))
if len(raw):
    duplicate_attempts=raw.duplicated(['model','protein_id','run_id','attempt_id']).sum()
    if duplicate_attempts: raise ValueError(f'{duplicate_attempts} duplicate raw attempt identifiers detected')
    # Ensure accepted rows appear in the same model/protein/run raw log. No chemistry assumptions.
    mismatches=[]
    for (model,prot,run),group in raw.groupby(['model','protein_id','run_id']):
        saved=Counter(group.loc[group.saved_in_main_csv,'smiles'])
        exported=Counter(samples.loc[(samples.model==model)&(samples.protein_id==prot)&(samples.run_id==run),'smiles'])
        if saved!=exported: mismatches.append((model,prot,run,len(saved),len(exported)))
    if mismatches:
        print('WARNING: raw accepted-sample flags do not match main CSV:',len(mismatches))
        display(pd.DataFrame(mismatches,columns=['model','protein','run','unique_raw_accepted','unique_exported']).head(12))
    print('Raw attempts:',len(raw),'| main accepted flags:',int(raw.saved_in_main_csv.sum()))
    display(raw.groupby('model').agg(attempts=('smiles','size'),accepted=('saved_in_main_csv','sum')))
else:
    print('No raw-attempt logs found; Method B will be unavailable rather than guessed.')

expected_raw_groups={item for item in expected_groups if item[0] in PYTORCH_MODELS}
observed_raw_groups=set(map(tuple,raw[['model','protein_id','run_id']].drop_duplicates().itertuples(index=False,name=None)))
absent_raw=sorted(expected_raw_groups-observed_raw_groups)
if absent_raw:print('WARNING: Missing raw-attempt groups:',len(absent_raw),'first:',absent_raw[:6])
if REQUIRE_TEN_COMPLETE_RUNS and absent_raw:
    raise ValueError('Raw-attempt logs missing for some PyTorch model/protein/run groups; '
                     'the independent validity comparison would be incomplete. '
                     f'First missing: {absent_raw[:5]}')
if any(item[0]=='nitesh_model' for item in expected_groups-observed_raw_groups):
    print('NOTE: MXNet nitesh_model raw attempts are optional; raw validity is N/A when absent.')
if REQUIRE_TEN_COMPLETE_RUNS and 'mismatches' in globals() and mismatches:
    raise ValueError('Raw flags do not match exported SMILES; resolve data mismatch first.')

## 7 · RDKit chemical validity and molecular properties

RDKit **parses and sanitizes** every distinct observed SMILES from *both* input tables. Empty strings, parser errors, and sanitation failures count as invalid in the raw-attempt denominator. No filtering happens before the raw-attempt metric. All descriptors, including SAS, are calculated **only for valid molecules**.

Descriptors replicate the mentor code's definitions: RDKit `ExactMolWt`, `MolLogP`, `QED.qed`, Lipinski counts and TPSA. SA scoring uses RDKit Contrib's `sascorer` implementation and reports its version path.

In [ ]:

import time
from functools import lru_cache
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. CHEMICAL VALIDITY: lightweight operation
# ---------------------------------------------------------

@lru_cache(maxsize=None)
def check_validity(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return {
            'canonical_smiles': None,
            'rdkit_valid': False,
            'invalid_reason': 'empty_or_missing_smiles'
        }

    try:
        m = Chem.MolFromSmiles(smiles.strip(), sanitize=False)

        if m is None:
            raise ValueError('SMILES parser returned None')

        if m.GetNumAtoms() == 0:
            raise ValueError('zero-atom molecule')

        Chem.SanitizeMol(m)

        canonical = Chem.MolToSmiles(
            m, canonical=True, isomericSmiles=True
        )

        return {
            'canonical_smiles': canonical,
            'rdkit_valid': True,
            'invalid_reason': None
        }

    except Exception as exc:
        return {
            'canonical_smiles': None,
            'rdkit_valid': False,
            'invalid_reason': f'{type(exc).__name__}: {str(exc)[:160]}'
        }


# ---------------------------------------------------------
# 2. MOLECULAR PROPERTIES: only for exported valid molecules
# ---------------------------------------------------------

@lru_cache(maxsize=None)
def calculate_properties(canonical_smiles):

    m = Chem.MolFromSmiles(canonical_smiles)

    mw = float(Descriptors.ExactMolWt(m))
    lp = float(Crippen.MolLogP(m))

    hd = int(Lipinski.NumHDonors(m))
    ha = int(Lipinski.NumHAcceptors(m))
    rb = int(Lipinski.NumRotatableBonds(m))

    tpsa = float(Descriptors.TPSA(m))
    qed = float(QED.qed(m))

    # Most expensive descriptor: calculate only when needed
    sas = float(sascorer.calculateScore(m))

    violations = sum([
        not (lp < 5),
        not (mw < 500),
        not (hd < 5),
        not (ha < 10)
    ])

    return {
        'MW': mw,
        'logP': lp,
        'QED': qed,
        'HD': hd,
        'HA': ha,
        'RB': rb,
        'TPSA': tpsa,
        'SAS': sas,
        'lipinski_violations': violations,
        'lipinski_0_violations': int(violations == 0)
    }


# ---------------------------------------------------------
# 3. COLLECT UNIQUE SMILES
# ---------------------------------------------------------

start_time = time.time()

all_keys = pd.unique(
    pd.concat(
        [samples['smiles'], raw['smiles']],
        ignore_index=True
    )
)

print(f'Total distinct SMILES representations: {len(all_keys):,}')
print('Stage 1: Checking RDKit validity...')


# ---------------------------------------------------------
# 4. CHECK VALIDITY
# ---------------------------------------------------------

validity_cache = {}

for sm in tqdm(all_keys, desc='RDKit validity'):
    validity_cache[sm] = check_validity(sm)

valid_count = sum(
    x['rdkit_valid'] for x in validity_cache.values()
)

print(f'\nRDKit valid: {valid_count:,}/{len(all_keys):,}')


# ---------------------------------------------------------
# 5. FIND UNIQUE CANONICAL EXPORTED MOLECULES
# ---------------------------------------------------------

exported_canonical = samples['smiles'].map(
    lambda s: validity_cache[s]['canonical_smiles']
)

unique_valid_canonical = pd.unique(
    exported_canonical.dropna()
)

print(
    f'\nUnique valid exported canonical molecules: '
    f'{len(unique_valid_canonical):,}'
)

print('Stage 2: Calculating molecular properties...')
print('This stage includes SAS, which is computationally expensive.')


# ---------------------------------------------------------
# 6. CALCULATE PROPERTIES
# ---------------------------------------------------------

property_cache = {}

for sm in tqdm(
    unique_valid_canonical,
    desc='Molecular properties'
):
    property_cache[sm] = calculate_properties(sm)

print(
    f'\nProperties calculated for '
    f'{len(property_cache):,} unique molecules.'
)


# ---------------------------------------------------------
# 7. ATTACH RESULTS TO EXPORTED SAMPLES
# ---------------------------------------------------------

DEFAULT_PROPERTIES = {
    'MW': np.nan,
    'logP': np.nan,
    'QED': np.nan,
    'HD': np.nan,
    'HA': np.nan,
    'RB': np.nan,
    'TPSA': np.nan,
    'SAS': np.nan,
    'lipinski_violations': np.nan,
    'lipinski_0_violations': np.nan
}

def get_sample_chemistry(sm):
    validity = validity_cache[sm]

    result = {
        **validity,
        **DEFAULT_PROPERTIES
    }

    canonical = validity['canonical_smiles']

    if canonical is not None:
        result.update(property_cache[canonical])

    return result


def attach_sample_chem(frame):
    fields = pd.DataFrame(
        [get_sample_chemistry(s) for s in frame.smiles],
        index=frame.index
    )

    return pd.concat([frame.copy(), fields], axis=1)


samples = attach_sample_chem(samples)


# ---------------------------------------------------------
# 8. ATTACH ONLY VALIDITY TO RAW ATTEMPTS
# ---------------------------------------------------------

def attach_raw_validity(frame):

    fields = pd.DataFrame(
        [validity_cache[s] for s in frame.smiles],
        index=frame.index
    )

    return pd.concat([frame.copy(), fields], axis=1)


if len(raw):
    raw = attach_raw_validity(raw)


# ---------------------------------------------------------
# 9. SUMMARY
# ---------------------------------------------------------

print('\nRDKit validity summary:')

display(
    samples.groupby('model').agg(
        exported=('smiles', 'size'),
        rdkit_valid=('rdkit_valid', 'sum'),
        distinct_canonical=('canonical_smiles', 'nunique')
    )
)

print('\nInvalid molecule examples:')

display(
    samples.loc[
        ~samples.rdkit_valid,
        ['model', 'protein_id', 'smiles', 'invalid_reason']
    ].head(8)
)

print(f'\nTotal Section 7 execution time: {(time.time()-start_time)/60:.2f} minutes')


## 8 · Two validity methods, uniqueness, and provenance checks

**Method A** is the mentor's exported-yield formula. **Method B** uses every recorded attempt and is reported only where the raw logs exist. The extra `rdkit_valid_exported_pct` measures the quality of the already filtered CSV, **not generator-wide validity**.

The three pretrained PyTorch checkpoints are unconditional and their per-protein IDs are repeated sampling slots. The new fine-tuned PyTorch checkpoint does consume the saved protein embedding per protein; it is genuinely conditioned. The MXNet `nitesh_model` uses its saved protein embeddings during generation, according to the provided generator.

In [ ]:
def safe_pct(n,d): return 100.0*n/d if d else np.nan

def metric_group(gr, model, prot, run):
    exported=len(gr)
    good=gr[gr.rdkit_valid]
    n_good=len(good)
    u_raw=gr.smiles.nunique()  # mentor counts raw exported strings, including any invalid rows
    u_canon=good.canonical_smiles.nunique()
    raw_grp=raw[(raw.model==model)&(raw.protein_id==prot)&(raw.run_id==run)] if len(raw) else raw
    n_attempts=len(raw_grp)
    n_raw_valid=int(raw_grp.rdkit_valid.sum()) if n_attempts else 0
    n_ended_valid=int((raw_grp.rdkit_valid & raw_grp.decode_status.eq('ended')).sum()) if n_attempts else 0
    metrics={
        'model':model,'protein_id':prot,'run_id':int(run),'exported_rows':exported,
        'rdkit_valid_exported_count':n_good,
        'rdkit_valid_exported_pct':safe_pct(n_good,exported),
        'mentor_validity_pct':safe_pct(exported,SAMPLE_BUDGET),
        'mentor_unique_raw_count':u_raw,
        'mentor_uniqueness_pct':safe_pct(u_raw,SAMPLE_BUDGET),
        'canonical_unique_count':u_canon,
        'canonical_uniqueness_valid_pct':safe_pct(u_canon,n_good),
        'raw_attempt_count':n_attempts if n_attempts else np.nan,
        'raw_rdkit_valid_count':n_raw_valid if n_attempts else np.nan,
        'independent_validity_pct':safe_pct(n_raw_valid,n_attempts),
        'raw_ended_and_valid_count':n_ended_valid if n_attempts else np.nan,
        'raw_ended_valid_pct':safe_pct(n_ended_valid,n_attempts),
        'raw_failure_count':(n_attempts-n_raw_valid) if n_attempts else np.nan,
        'mean_QED_valid':good.QED.mean() if n_good else np.nan,
        'mean_SAS_valid':good.SAS.mean() if n_good else np.nan,
        'mean_logP_valid':good.logP.mean() if n_good else np.nan,
        'mean_MW_valid':good.MW.mean() if n_good else np.nan,
        'lipinski_zero_violations_pct_valid':safe_pct(int(good.lipinski_0_violations.sum()),n_good),
    }
    tests={'logP':good.logP<5,'MW':good.MW<500,'HD':good.HD<5,'HA':good.HA<10,
           'RB':good.RB<10,'TPSA':good.TPSA<140,'SAS':good.SAS<6}
    # Exact reference formula: 100 * (nunique(raw)/N) * (n_pass/N).
    metrics['ref_QED']=good.QED.sum()/SAMPLE_BUDGET
    for key,cond in tests.items():
        p=int(cond.sum())
        metrics[f'{key}_pass_count']=p
        metrics[f'ref_{key}_pct']=100*(u_raw/SAMPLE_BUDGET)*(p/SAMPLE_BUDGET)
        metrics[f'standard_{key}_pct_valid']=safe_pct(p,n_good)
        metrics[f'standard_{key}_pct_exported']=safe_pct(p,exported)
    return metrics

per_group_rows=[]
for (model,prot,run),group in samples.groupby(['model','protein_id','run_id'],sort=True):
    per_group_rows.append(metric_group(group,model,prot,run))
per_group=pd.DataFrame(per_group_rows)
print('Validity methods (mean of protein/run groups; N/A if no raw attempts):')
display(per_group.groupby('model')[['mentor_validity_pct','independent_validity_pct',
    'rdkit_valid_exported_pct','mentor_uniqueness_pct','canonical_uniqueness_valid_pct']].mean().round(2))
if per_group['independent_validity_pct'].isna().any():
    print('NOTE: Some raw-attempt validity values are N/A. These are not zero or 100%.')

## 9 · Novelty (optional; never invent a training set)

Conventional novelty here means **canonical RDKit-valid generated molecule not present among verified training molecules**. It is not the same as chemical uniqueness. You must supply the actual train/pretraining ligand reference appropriate to the model and document its provenance. If the models trained on different corpora, a *single* novelty reference can be shown as a common baseline, but should not be called model-training novelty. With no verified reference file, novelty = **N/A**.

In [ ]:
def load_reference_smiles(path,smiles_column):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower() in ('.csv','.tsv'):
        sep='\t' if p.suffix.lower()=='.tsv' else ','
        df=pd.read_csv(p,sep=sep,keep_default_na=False)
        col=_get_column(df,smiles_column,('Smiles','smiles','SMILES'))
        if col is None: raise ValueError(f'Novelty reference {p}: missing SMILES column; columns={list(df.columns)}')
        strings=df[col].astype(str).tolist()
    elif p.suffix.lower() in ('.smi','.txt'):
        strings=[line.strip().split()[0] for line in p.read_text().splitlines() if line.strip() and not line.lstrip().startswith('#')]
    else:
        raise ValueError('Novelty source requires CSV, TSV, SMI or TXT with known SMILES content')
    valid={}
    for string in strings:
        x=check_validity(string)
        if x['rdkit_valid']: valid[x['canonical_smiles']]=1
    print('Reference rows:',len(strings),'| distinct valid canonical:',len(valid))
    if not valid: raise ValueError('No valid reference ligands found')
    return set(valid)

reference_by_model={}
if NOVELTY_REFERENCE_BY_MODEL:
    for model,path in NOVELTY_REFERENCE_BY_MODEL.items():
        if model not in MODELS:raise ValueError(f'Unknown model for novelty: {model}')
        reference_by_model[model]=load_reference_smiles(path,NOVELTY_SMILES_COLUMN)
elif NOVELTY_REFERENCE_FILE:
    shared=load_reference_smiles(NOVELTY_REFERENCE_FILE,NOVELTY_SMILES_COLUMN)
    reference_by_model={model:shared for model in MODELS}

samples['is_novel']=pd.Series(pd.NA,index=samples.index,dtype='boolean')
novelty_rows=[]
for (model,prot,run),g in samples.groupby(['model','protein_id','run_id']):
    refset=reference_by_model.get(model)
    if refset is None:
        novelty_rows.append({'model':model,'protein_id':prot,'run_id':run,
            'novel_unique_count':np.nan,'novelty_unique_valid_pct':np.nan,'novelty_exported_pct':np.nan})
        continue
    idx=g[g.rdkit_valid].index
    samples.loc[idx,'is_novel']=samples.loc[idx,'canonical_smiles'].map(lambda s: s not in refset).astype(bool)
    valid=samples.loc[idx]
    unique=valid.drop_duplicates('canonical_smiles')
    n_novel=int(unique.is_novel.fillna(False).sum())
    novelty_rows.append({'model':model,'protein_id':prot,'run_id':run,
        'novel_unique_count':n_novel,
        'novelty_unique_valid_pct':safe_pct(n_novel,len(unique)),
        'novelty_exported_pct':safe_pct(int(valid.is_novel.fillna(False).sum()),len(g))})
per_group=per_group.merge(pd.DataFrame(novelty_rows),on=['model','protein_id','run_id'],validate='one_to_one')
if not reference_by_model:
    print('NOVELTY: N/A. Supply NOVELTY_REFERENCE_FILE or NOVELTY_REFERENCE_BY_MODEL with verified training ligands.')
else:
    print('NOVELTY available for models:',list(reference_by_model))

## 10 · Ames-mutagenicity labels (RF surrogate or pkCSM export)

**CTDDG Table 3 uses pkCSM predictions**, not laboratory Ames measurements and not the Random Forest used to annotate the pretraining ChEMBL molecules. The notebook can:

- Score valid unique SMILES with a **trusted local Random Forest**, using Morgan **radius 2, 2048 bits** if and only if that matches your saved model's actual training features. It reports RF-predicted Ames toxicity separately and marks its feature/label provenance as unverified until you set `RF_FEATURE_CONTRACT_VERIFIED=True` after auditing training code.
- Read **previously acquired pkCSM Ames predictions** from a CSV; it will not send requests or assume predicted values.
- Leave toxicity **unavailable** if neither source exists.

**Unknown labels remain `NaN`**, not non-toxic. This is Ames *mutagenicity*, not general clinical or environmental safety.

In [ ]:
import joblib

def parse_ames_label(value):
    if pd.isna(value):return np.nan
    s=str(value).strip().lower().replace('_','-')
    if s in ('1','1.0','yes','true','positive','toxic','mutagenic','mutagenicity-positive'):return 1.0
    if s in ('0','0.0','no','false','negative','non-toxic','nontoxic','non-mutagenic'):return 0.0
    return np.nan

pkcsm_map={}
if PKCSM_AMes_CSV is not None:
    p=Path(PKCSM_AMes_CSV)
    if not p.is_file(): raise FileNotFoundError(p)
    df=pd.read_csv(p,keep_default_na=False)
    if PKCSM_SMILES_COLUMN not in df or PKCSM_LABEL_COLUMN not in df:
        raise ValueError(f'pkCSM export must contain {PKCSM_SMILES_COLUMN} and {PKCSM_LABEL_COLUMN}')
    for sm,label in zip(df[PKCSM_SMILES_COLUMN],df[PKCSM_LABEL_COLUMN]):
        chem=check_validity(str(sm)); y=parse_ames_label(label)
        if chem['rdkit_valid'] and not pd.isna(y):
            key=chem['canonical_smiles']
            if key in pkcsm_map and pkcsm_map[key]!=y:raise ValueError(f'Conflicting pkCSM Ames labels for {key}')
            pkcsm_map[key]=y
    print('Validated pkCSM labels for',len(pkcsm_map),'distinct canonical SMILES')

rf_map={}; rf_provenance='unavailable'
if AMES_MODE=='rf_if_available':
    if RF_MODEL_FILE.is_file():
        print('Reading trusted local Ames RF:',RF_MODEL_FILE)
        rf=joblib.load(RF_MODEL_FILE) # joblib/pickle files can execute code. TRUST ONLY YOUR OWN MODEL.
        if not hasattr(rf,'predict') or not hasattr(rf,'classes_'):
            raise ValueError('Saved Ames predictor must support predict and classes_')
        classes=list(rf.classes_)
        if RF_POSITIVE_LABEL not in classes:
            raise ValueError(f'Positive label {RF_POSITIVE_LABEL!r} not among trained classes {classes!r}')
        if RF_FINGERPRINT_MODE!='morgan_bits':
            raise NotImplementedError('Only Morgan bit fingerprints are configured: audit the training pipeline first')
        if getattr(rf,'n_features_in_',RF_BITS)!=RF_BITS:
            raise ValueError(f'RF expects {rf.n_features_in_} features, but configured {RF_BITS} bits')
        uniques=sorted(set(samples.loc[samples.rdkit_valid,'canonical_smiles']))
        from rdkit.Chem import rdFingerprintGenerator
        fp_gen=rdFingerprintGenerator.GetMorganGenerator(radius=RF_RADIUS,fpSize=RF_BITS)
        batch_size=4096
        for start in range(0,len(uniques),batch_size):
            portion=uniques[start:start+batch_size]
            X=np.zeros((len(portion),RF_BITS),dtype=np.uint8)
            for i,s in enumerate(portion):
                fp=fp_gen.GetFingerprint(Chem.MolFromSmiles(s))
                DataStructs.ConvertToNumpyArray(fp,X[i])
            y=rf.predict(X)
            rf_map.update({s:float(v==RF_POSITIVE_LABEL) for s,v in zip(portion,y)})
        rf_provenance=('verified model contract' if RF_FEATURE_CONTRACT_VERIFIED else
                       'UNVERIFIED feature/class-label semantics; exploratory RF only')
        print(f'RF Ames scored: {len(rf_map):,} canonical molecules; {rf_provenance}')
    else:
        print('RF model missing:',RF_MODEL_FILE, '| RF Ames score N/A.')
elif AMES_MODE=='pkcsm_csv':
    if not pkcsm_map: print('No pkCSM labels configured: Ames score N/A.')
elif AMES_MODE=='none':
    print('Ames predictions disabled by configuration.')
else:
    raise ValueError('AMES_MODE must be rf_if_available, pkcsm_csv or none')

samples['ames_rf_toxic']=pd.to_numeric(samples.canonical_smiles.map(rf_map),errors='coerce')
samples['ames_pkcsm_toxic']=pd.to_numeric(samples.canonical_smiles.map(pkcsm_map),errors='coerce')
# Primary source for internal comparison; paper comparison requires pkCSM.
primary_col='ames_pkcsm_toxic' if AMES_MODE=='pkcsm_csv' else 'ames_rf_toxic'
samples['ames_toxic_primary']=samples[primary_col]
print('Predicted Ames coverage by model (fraction of exported rows):')
display(samples.groupby('model').agg(
    total=('smiles','size'),rf_scored=('ames_rf_toxic','count'),pkcsm_scored=('ames_pkcsm_toxic','count')))

## 11 · Calculate Ames rates with explicit coverage and denominators

In [ ]:
toxic_rows=[]
for (model,prot,run),g in samples.groupby(['model','protein_id','run_id']):
    # Exclude invalid from predictor denominator, but NEVER treat unscored as non-toxic.
    valid=g[g.rdkit_valid]
    r={'model':model,'protein_id':prot,'run_id':run,'valid_exported':len(valid),
       'exported_total':len(g)}
    for source,column in [('rf','ames_rf_toxic'),('pkcsm','ames_pkcsm_toxic')]:
        v=valid[column].dropna()
        r[f'{source}_scored_count']=len(v)
        r[f'{source}_toxic_count']=int((v==1).sum())
        r[f'{source}_coverage_valid_pct']=safe_pct(len(v),len(valid))
        r[f'{source}_toxic_pct_scored']=safe_pct(int((v==1).sum()),len(v))
        r[f'{source}_toxic_pct_exported']=safe_pct(int((v==1).sum()),len(g)) if len(v)==len(valid) and len(valid)==len(g) else np.nan
    toxic_rows.append(r)
toxic_groups=pd.DataFrame(toxic_rows)
per_group=per_group.merge(toxic_groups,on=['model','protein_id','run_id'],validate='one_to_one')
print('Model-level Ames coverage and percentages (rates only among scored molecules):')
for model,g in samples.groupby('model'):
    for mode,col in [('RF','ames_rf_toxic'),('pkCSM','ames_pkcsm_toxic')]:
        sub=g[g.rdkit_valid][col].dropna()
        print(f'{model:27} {mode:6} {len(sub):6d} scored | '
              f'{safe_pct(int((sub==1).sum()),len(sub)):.2f}% mutagenic' if len(sub)
              else f'{model:27} {mode:6} N/A: no scored molecules')

## 12 · Per-run and overall comparison (respecting the paper's hierarchy)

The paper averages the **per-protein percentages within each run**, then reports the mean and population SD of the **10 run means**. This notebook follows that aggregation for mentor-style Table 1 columns; it also exports pooled counts for conventional interpretation. When you only have **one run**, it does not invent `± 0.00`.

In [ ]:
SUMMARY_METRICS={
  'mentor_validity_pct':'Mentor validity (%)',
  'independent_validity_pct':'Independent raw RDKit validity (%)',
  'rdkit_valid_exported_pct':'RDKit validity among exported (%)',
  'mentor_uniqueness_pct':'Mentor uniqueness (%)',
  'canonical_uniqueness_valid_pct':'Canonical uniqueness among valid (%)',
  'novelty_unique_valid_pct':'Novelty among unique valid (%)',
  'rf_toxic_pct_scored':'RF-predicted Ames mutagenic (%)',
  'pkcsm_toxic_pct_scored':'pkCSM-predicted Ames mutagenic (%)',
  'mean_QED_valid':'QED mean among valid',
  'mean_SAS_valid':'SAS mean among valid',
  'lipinski_zero_violations_pct_valid':'Zero Lipinski violations among valid (%)',
}

per_run=(per_group.groupby(['model','run_id'],as_index=False)
         .agg({**{k:'mean' for k in SUMMARY_METRICS},
               'protein_id':'nunique','exported_rows':'sum','rdkit_valid_exported_count':'sum',
               'canonical_unique_count':'sum','raw_attempt_count':lambda s:s.sum(min_count=1),
               'raw_rdkit_valid_count':lambda s:s.sum(min_count=1),
               'rf_scored_count':'sum','rf_toxic_count':'sum','pkcsm_scored_count':'sum','pkcsm_toxic_count':'sum'}))
per_run=per_run.rename(columns={'protein_id':'protein_slots'})

def mean_sd_str(values,decimal=2):
    v=pd.to_numeric(pd.Series(values),errors='coerce').dropna()
    if not len(v): return 'N/A'
    if len(v)==1: return f'{v.iloc[0]:.{decimal}f} (1 run; SD N/A)'
    return f'{v.mean():.{decimal}f} ± {v.std(ddof=0):.2f}'

overall_rows=[]
for model in MODELS:
    g=per_group[per_group.model==model]
    runs=per_run[per_run.model==model]
    orig=samples[samples.model==model]
    raw_g=raw[raw.model==model] if len(raw) else raw
    r={'model':model, 'protein_conditioned':MODEL_IS_PROTEIN_CONDITIONAL[model],
       'runs_observed':runs.run_id.nunique(),
       'proteins_observed':g.protein_id.nunique(),
       'samples_exported':len(orig),
       'valid_exported_count':int(orig.rdkit_valid.sum()) if len(orig) else 0,
       'unique_canonical_all_saved_count':orig.canonical_smiles.nunique(),
       'attempts_recorded':len(raw_g) if len(raw_g) else np.nan,
       'valid_raw_attempts':int(raw_g.rdkit_valid.sum()) if len(raw_g) else np.nan,
       'ames_rf_scored':int(orig.ames_rf_toxic.count()),
       'ames_pkcsm_scored':int(orig.ames_pkcsm_toxic.count())}
    for key,label in SUMMARY_METRICS.items():
        r[label]=mean_sd_str(runs[key])
    overall_rows.append(r)
overall=pd.DataFrame(overall_rows)
display(overall)

## 13 · Reproduce the **structure** of published CTDDG Table 1

In [ ]:
# Paper values transcribed from the supplied CTDDG(2).pdf, page 11, Table 1.
TABLE1_METRICS=[
    ('Validity (%)','mentor_validity_pct','91.7 ± 0.22','88.4 ± 3.29','94.4 ± 0.32'),
    ('Uniqueness (%)','mentor_uniqueness_pct','91.7 ± 0.22','58.1 ± 4.13','94.4 ± 0.32'),
    ('LogP < 5 (%)','ref_logP_pct','68.6 ± 0.38','51.4 ± 4.31','74.8 ± 0.33'),
    ('Exact MW < 500 (%)','ref_MW_pct','73.5 ± 0.47','51.4 ± 4.31','71.1 ± 0.32'),
    ('H-bond donors < 5 (%)','ref_HD_pct','84.6 ± 0.25','51.4 ± 4.31','87.9 ± 0.4'),
    ('H-bond acceptors < 10 (%)','ref_HA_pct','84.5 ± 0.28','51.4 ± 4.31','87.2 ± 0.43'),
    ('Rotatable bonds < 10 (%)','ref_RB_pct','80.3 ± 0.40','51.4 ± 4.31','79.9 ± 0.38'),
    ('TPSA < 140 (%)','ref_TPSA_pct','83.5 ± 0.23','51.4 ± 4.31','84.1 ± 0.36'),
    ('QED (absolute)','ref_QED','0.55 ± 0.00','0.44 ± 0.02','0.51 ± 0.0'),
    ('SAS < 6 (%)','ref_SAS_pct','84.7 ± 0.23','51.4 ± 4.31','88.6 ± 0.37'),
]
table1=[]
for label,k,cdgcn,ssvae,ctddg in TABLE1_METRICS:
    row={'Metric':label,'Published CDGCN':cdgcn,'Published SSVAE':ssvae,'Published CTDDG':ctddg}
    for model in MODELS:
        sub=per_group[per_group.model==model].groupby('run_id')[k].mean() if len(per_group) else pd.Series(dtype=float)
        row[model+' (mentor-compatible)']=mean_sd_str(sub,2 if k=='ref_QED' else 2)
    table1.append(row)
table1=pd.DataFrame(table1)
print('Paper Table 1 structure — mentor-compatible calculations; published columns are reference only.')
display(table1)
print('Separate independent validity (NOT the same estimand):')
display(overall[['model','Independent raw RDKit validity (%)','RDKit validity among exported (%)']])

## 14 · Table 3: published PDB-target values vs your predicted Ames rates

Your generated CSVs use `protein1`, `protein2`, ... **not verified PDB identifiers**. Without an externally verified mapping for `4nos`, `2nru`, `1lqf`, `1k2r`, `1cqp`, there is no defensible way to assign a `proteinN` folder to one of the paper's five targets. This notebook therefore always exports a **104-slot internal toxicity table**, but only populates the five-target paper reproduction when you provide a verified `PDB_TO_PROTEIN_ID` mapping. **RF and pkCSM are separate columns**.

In [ ]:
PAPER_TABLE3={
 '4nos':('24.13 ± 4.52','17.5 ± 3.77','11.7 ± 4.13','19.84 ± 4.38'),
 '2nru':('26.89 ± 5.31','30.1 ± 11.17','11.7 ± 4.13','23.32 ± 4.52'),
 '1lqf':('23.12 ± 4.73','N/A','11.7 ± 4.13','20.26 ± 3.88'),
 '1k2r':('22.49 ± 4.00','17.9 ± 7.06','11.7 ± 4.13','20.02 ± 7.29'),
 '1cqp':('19.88 ± 4.11','36.2 ± 8.36','11.7 ± 4.13','13.17 ± 5.31'),
 'Avg':('23.30 ± 4.53','25.43 ± 6.4','11.7 ± 4.13','19.32 ± 5.08'),
}
if PDB_TO_PROTEIN_ID and (set(PDB_TO_PROTEIN_ID)!=set(PAPER_TABLE3)-{'Avg'}):
    raise ValueError('Provide exactly the five required PDB→proteinN mappings, not a subset.')
if len(set(PDB_TO_PROTEIN_ID.values())) != len(PDB_TO_PROTEIN_ID):
    raise ValueError('Two PDB targets map to the same protein slot; verify the mapping')

paper3=[]
for pdb,(cdgcn,pocket,ssvae,ctddg) in PAPER_TABLE3.items():
    row={'PDB target':pdb,'Published CDGCN (pkCSM)':cdgcn,
         'Published Pocket2mol (pkCSM)':pocket,'Published SSVAE (pkCSM)':ssvae,
         'Published CTDDG (pkCSM)':ctddg,
         'Mapped test slot':PDB_TO_PROTEIN_ID.get(pdb,'N/A') if pdb!='Avg' else '5-target mean'}
    for model in MODELS:
        if pdb=='Avg':
            for source in ('rf','pkcsm'):
                cols=[]
                for target,prot in PDB_TO_PROTEIN_ID.items():
                    r=per_group[(per_group.model==model)&(per_group.protein_id==prot)].groupby('run_id')[f'{source}_toxic_pct_scored'].mean()
                    if len(r) and r.notna().any(): cols.append(float(r.mean()))
                row[model+' '+source]=f'{np.mean(cols):.2f}% (target mean; SD N/A)' if len(cols)==5 else 'N/A'
        else:
            prot=PDB_TO_PROTEIN_ID.get(pdb)
            for source in ('rf','pkcsm'):
                r=per_group[(per_group.model==model)&(per_group.protein_id==prot)].groupby('run_id')[f'{source}_toxic_pct_scored'].mean() if prot else pd.Series(dtype=float)
                row[model+' '+source]=mean_sd_str(r)
    paper3.append(row)
table3=pd.DataFrame(paper3)
display(table3)
if not PDB_TO_PROTEIN_ID:
    print('PAPER TABLE 3 STATUS: published numbers displayed, experimental PDB rows N/A until verified mapping.')
print('Internal per-protein-slot RF/pkCSM results (all observed slots):')
display(per_group[['model','protein_id','run_id','rf_toxic_pct_scored','rf_coverage_valid_pct',
                  'pkcsm_toxic_pct_scored','pkcsm_coverage_valid_pct']].head(12))

### Table 3 supplement · RF-predicted Ames mutagenicity across all 104 sample slots

The paper's Table 3 refers to **five identified PDB targets** scored by pkCSM. Your **pretrained** PyTorch model slots are unconditioned and the mapping from `proteinN` to those five PDB targets is unverified. The following table is therefore a **separate 104-slot, 10-run Random Forest baseline**, not a replication of Table 3's target-specific protocol.

In [ ]:
rf_all_104=[]
for model in MODELS:
    subset=per_group[per_group.model.eq(model)]
    per= subset.groupby('run_id',sort=True)['rf_toxic_pct_scored'].mean()
    rf_all_104.append({'model':model,'test_slots':subset.protein_id.nunique(),
        'runs':per.count(),'RF Ames mutagenic (%) mean ± SD':mean_sd_str(per),
        'scored_valid_molecules':int(subset.rf_scored_count.sum()),
        'toxic_valid_molecules':int(subset.rf_toxic_count.sum()),
        'RF labels':'predicted (not experimental; not pkCSM)',
        'protein_conditioned':MODEL_IS_PROTEIN_CONDITIONAL[model]})
rf_overall_104=pd.DataFrame(rf_all_104)
print('Internal RF comparison across all 104 slots (NOT CTDDG Table 3 protocol):')
display(rf_overall_104)

## 15 · Table 2: target awareness from **external docking results only**

SMILES cannot establish binding affinity. If you have already docked generated ligands against the corresponding PDB target using SMINA, set `DOCKING_CSV` to a CSV with `model, protein_id, Run_ID, SMILES, binding_kcal_mol`. The optional file is evaluated without running docking again. It is **not comparable to the paper unless the correct five PDB structures, binding-site settings, and protein–ligand mappings are confirmed**. The paper's Table 4 CDK2-specific LD50/LC50 and molecular-dynamics results remain a separate case study.

In [ ]:
PAPER_TABLE2={
 '4nos':('99.10 ± 1.08','72.49 ± 6.34','11.97 ± 3.04','95.55 ± 2.56'),
 '2nru':('96.28 ± 2.15','95.25 ± 2.91','2.49 ± 1.19','93.33 ± 3.58'),
 '1lqf':('96.02 ± 1.51','N/A','4.23 ± 1.38','98.03 ± 2.09'),
 '1k2r':('98.57 ± 0.67','61.22 ± 11.1','19.04 ± 3.08','95.71 ± 2.92'),
 '1cqp':('85.00 ± 4.07','97.74 ± 1.67','2.49 ± 1.19','86.35 ± 3.43'),
 'Avg':('95.00 ± 1.90','81.68 ± 5.51','8.04 ± 1.98','93.80 ± 2.92'),
}
# Published Table 2, transcribed from PDF page 12; no experimental binding affinity assumed.
table2_published=pd.DataFrame([{'PDB target':k,'Published CDGCN':v[0],
 'Published Pocket2mol':v[1],'Published SSVAE':v[2],'Published CTDDG':v[3]}
 for k,v in PAPER_TABLE2.items()])
# Important: the uploaded paper actually prints 1k2r 61.22±1.11? verify manuscript typographic table
# (reproduction Table 2 is not computed from current samples in this notebook).
if DOCKING_CSV is None:
    print('Table 2 reproduction: N/A (no verified external SMINA docking measurements).')
    table2_experimental=pd.DataFrame(columns=['model','protein_id','run_id','docked_count','binding_below_minus7_pct'])
else:
    docking=pd.read_csv(DOCKING_CSV)
    expected={'model','protein_id','Run_ID','SMILES','binding_kcal_mol'}
    if not expected.issubset(docking.columns):raise ValueError(f'Docking CSV missing {expected-set(docking.columns)}')
    docking['binding_kcal_mol']=pd.to_numeric(docking.binding_kcal_mol,errors='coerce')
    docking['canonical_smiles']=docking['SMILES'].astype(str).map(lambda s:check_validity(s)['canonical_smiles'])
    if docking.binding_kcal_mol.isna().any():print('WARNING: some docking energies invalid and excluded')
    dr=docking.dropna(subset=['binding_kcal_mol','canonical_smiles'])
    allowed=samples[['model','protein_id','run_id','canonical_smiles']].dropna().drop_duplicates()
    dr=dr.rename(columns={'Run_ID':'run_id'}).merge(allowed,on=['model','protein_id','run_id','canonical_smiles'],how='inner',validate='many_to_one')
    table2_experimental=dr.groupby(['model','protein_id','run_id']).binding_kcal_mol.agg(
        docked_count='size',binding_below_minus7_pct=lambda x:100*(x< -7.0).mean()).reset_index()
    print('Docked results; denominator is scored/docked molecules, not unmeasured generated molecules.')
    display(table2_experimental.head())
display(table2_published)

## 16 · Figures and failure diagnostics (saved under Evaluation_Results)

In [ ]:
import matplotlib.pyplot as plt

if PLOT_RESULTS:
    order=list(MODELS)
    fig,axes=plt.subplots(1,2,figsize=(15,5.1))
    a=per_group.groupby('model').mentor_validity_pct.mean().reindex(order)
    b=per_group.groupby('model').independent_validity_pct.mean().reindex(order)
    for ax,data,title in [(axes[0],a,'Method A · mentor exported yield'),
                          (axes[1],b,'Method B · raw RDKit validity')]:
        ax.bar(range(len(order)),data.fillna(0).values)
        ax.set_xticks(range(len(order)),[
            'MXNet', 'Legacy CDGCN', 'Legacy CTDDG',
            'Clean CDGCN (5ep)', 'Fine-tuned CTDDG'
        ],rotation=25,ha='right')
        ax.set_title(title);ax.set_ylabel('Percent');ax.set_ylim(0,max(105,float(data.max())*1.1) if data.notna().any() else 105)
        for i,v in enumerate(data):ax.text(i, (float(v) if pd.notna(v) else 0)+1,
                                           f'{v:.1f}%' if pd.notna(v) else 'N/A',ha='center',fontsize=9)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'validity_methods.png',dpi=150,bbox_inches='tight')
    plt.show()
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for model,g in samples[samples.rdkit_valid].groupby('model'):
        axes[0].hist(g.QED.dropna(),bins=20,alpha=.45,label=model,density=True)
        axes[1].hist(g.SAS.dropna(),bins=20,alpha=.45,label=model,density=True)
    axes[0].set_title('QED · exported valid molecules');axes[1].set_title('SAS · exported valid molecules')
    for ax in axes:ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(OUTPUT_DIR/'qed_sas_distributions.png',dpi=150,bbox_inches='tight');plt.show()

if len(raw):
    print('Raw decoder status counts (do not confuse status with RDKit chemical validity):')
    display(raw.groupby(['model','decode_status'],dropna=False).size().rename('attempts').reset_index().head(25))
print('Invalid RDKit molecules by model and reason (top entries):')
display(samples.loc[~samples.rdkit_valid].groupby(['model','invalid_reason']).size().rename('count').reset_index().head(25))

## 17 · Export auditable results and reproducibility metadata

In [ ]:
exports={
 'overall_model_comparison.csv':overall,
 'rf_ames_all_104_slots.csv':rf_overall_104,
 'table1_published_vs_five_models.csv':table1,
 'table3_published_vs_five_models.csv':table3,
 'table2_published_reference.csv':table2_published,
 'table2_optional_docking.csv':table2_experimental,
 'per_protein_run_metrics.csv':per_group,
 'per_run_aggregates.csv':per_run,
 'ames_by_protein_run.csv':toxic_groups,
 'file_inventory.csv':inventory,
}
if WRITE_DETAILED_ROWS:
    exports['exported_molecules_with_descriptors.csv']=samples
    if len(raw): exports['raw_attempts_with_rdkit_validity.csv']=raw
for name,frame in exports.items():
    frame.to_csv(OUTPUT_DIR/name,index=False)

meta={'generated_root':str(GENERATED_ROOT),'model_folders':list(MODELS),
  'protein_conditioning_by_model':MODEL_IS_PROTEIN_CONDITIONAL,
 'sample_budget_per_run':SAMPLE_BUDGET,'expected_proteins':len(EXPECTED_PROTEIN_IDS),
 'paper_runs':PAPER_RUNS,'require_ten_complete_runs':REQUIRE_TEN_COMPLETE_RUNS,'observed_runs_by_model':per_run.groupby('model').run_id.nunique().to_dict(),
 'source_inventory_file':str(OUTPUT_DIR/'input_file_inventory.csv'),
 'paper_validity_method':'100*exported_rows/budget (mentor yield proxy)',
 'independent_validity_method':'100*RDKit_sanitized_raw_attempts/all_logged_attempts',
 'missing_raw_attempts_not_imputed':True,
 'novelty_reference_file':str(NOVELTY_REFERENCE_FILE) if NOVELTY_REFERENCE_FILE else None,
 'novelty_reference_by_model':{k:str(v) for k,v in NOVELTY_REFERENCE_BY_MODEL.items()},
 'ames_mode':AMES_MODE,'rf_path':str(RF_MODEL_FILE),'rf_provenance':rf_provenance,
 'pkcsm_labels_supplied':len(pkcsm_map),'pdb_to_test_mapping':PDB_TO_PROTEIN_ID,
 'docking_file':str(DOCKING_CSV) if DOCKING_CSV else None,
 'python_version':sys.version,'rdkit_version':rdBase.rdkitVersion,
 'pandas_version':pd.__version__,'sas_path':str(sas_path),
 'warnings':[
    'MXNet raw attempt validity requires the new MXNet generator raw-attempt logs',
    'Three pretrained PyTorch generators are unconditional; their protein labels are sample slots',
     'Fine-tuned PyTorch CTDDG uses protein embeddings and the source run is marked exploratory_leaked_pretraining',
    '10 complete independent runs required for the paper-style mean ± SD',
    'RF Ames percentages are not pkCSM nor laboratory Ames measurements',
    'no Table 3 PDB mapping may be invented',
 ]}
(OUTPUT_DIR/'evaluation_provenance.json').write_text(json.dumps(meta,indent=2,default=str))
print('Saved outputs in:',OUTPUT_DIR)
display(pd.DataFrame([{'filename':p.name,'size_KB':round(p.stat().st_size/1024,1)}
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))

## 18 · Final interpretation and methodological limitations

**For a defensible reproduction of the published CTDDG numbers**, these limitations must be resolved before making comparative claims:

1. **Method A is not true generator-wide validity.** Generation deliberately keeps sampling until accepted molecules reach the budget. `100%` means the exported file filled its quota. Method B can only be recovered from original raw attempts; MXNet's current main CSVs alone cannot provide it.
2. **Paper Table 1 uses 10 independent runs over 104 protein-conditioned tests.** If your generation files have one run, this notebook prints `1 run; SD N/A` and does not claim the paper's `±` values are replicated.
3. **Three PyTorch pretrained models are unconditional.** Their 104 sampling slots do **not** establish target awareness. The fine-tuned PyTorch model uses protein embeddings, but target-specific paper comparisons still need verified PDB mapping and consistent methodology. Use model-stage labels when presenting findings.
4. **Ames vs toxicity:** an RF Ames classification is a model prediction of one mutagenicity endpoint. The paper's Table 3 uses pkCSM Ames predictions for five named targets; neither is experimental Ames lab testing.
5. **Novelty** requires the verified reference ligand corpus, canonicalization policy and leakage audit; without it, novelty is N/A.
6. **PDB target mapping and docking** are required before Tables 2/3 can be compared target-by-target. No sequential mapping between `proteinN` and `4nos`, etc. is inferred.
7. **SA/QED/drug-likeness descriptors** are computed for sanitizable molecules. The mentor's uniqueness-weighted pass fractions are exported separately from conventional property pass rates; the paper does not fully specify every implementation detail.

A strong research report should present **both methods**, raw sample counts, class-prediction coverage and caveats, instead of reporting a single misleading validity percentage.